# 📱 Mobile Phone Price Classification - Demo

This notebook demonstrates the complete ML pipeline for predicting mobile phone prices.

## Table of Contents
1. [Setup & Imports](#1-setup)
2. [Load & Explore Data](#2-load-data)
3. [Preprocessing](#3-preprocess)
4. [Train Models](#4-train)
5. [Evaluate Models](#5-evaluate)
6. [Make Predictions](#6-predict)

## 1. Setup & Imports

In [ ]:
# Import necessary libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Load & Explore Data

In [ ]:
# Load the dataset
# Replace with your actual data path
data_path = '../data/mobile_phone_data.csv'

try:
    df = pd.read_csv(data_path)
    print(f"Dataset shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print("Data file not found. Using sample data for demonstration.")
    # Sample data for demonstration
    np.random.seed(42)
    n_samples = 1000
    df = pd.DataFrame({
        'battery_power': np.random.randint(500, 2000, n_samples),
        'ram': np.random.randint(256, 4000, n_samples),
        'storage': np.random.randint(4, 128, n_samples),
        'camera_mp': np.random.randint(0, 20, n_samples),
        'price_range': np.random.randint(0, 4, n_samples)  # Target: 0-3
    })
    print(f"Sample dataset created: {df.shape}")
    display(df.head())

In [ ]:
# Data exploration
print("Dataset Info:")
df.info()

print("\n\nStatistical Summary:")
display(df.describe())

print("\n\nTarget Distribution:")
print(df['price_range'].value_counts().sort_index())

## 3. Preprocessing

In [ ]:
# Separate features and target
X = df.drop('price_range', axis=1)
y = df['price_range']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Train Models

In [ ]:
# Train multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'accuracy': accuracy, 'predictions': y_pred}
    
    print(f"  Accuracy: {accuracy:.4f}")

print("\n✅ Training complete!")

## 5. Evaluate Models

In [ ]:
# Compare model performance
print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)

for name, result in results.items():
    print(f"\n{name}:")
    print(f"  Accuracy: {result['accuracy']:.4f}")

In [ ]:
# Best model
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_model = results[best_model_name]['model']
best_predictions = results[best_model_name]['predictions']

print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {results[best_model_name]['accuracy']:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High', 'V.High'],
            yticklabels=['Low', 'Medium', 'High', 'V.High'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_test, best_predictions, 
                          target_names=['Low', 'Medium', 'High', 'V.High']))

## 6. Make Predictions

In [ ]:
# Make prediction on new data
new_phone = pd.DataFrame({
    'battery_power': [1500],
    'ram': [3000],
    'storage': [64],
    'camera_mp': [12]
})

# Scale and predict
new_phone_scaled = scaler.transform(new_phone)
prediction = best_model.predict(new_phone_scaled)

price_labels = {0: 'Low Price', 1: 'Medium Price', 2: 'High Price', 3: 'Very High Price'}

print("New Phone Specifications:")
display(new_phone)
print(f"\n🎯 Predicted Price Range: {price_labels[prediction[0]]}")

In [ ]:
# Save the model (optional)
import pickle

model_path = '../models/trained_model.pkl'
scaler_path = '../models/scaler.pkl'

# Create models directory if not exists
import os
os.makedirs('../models', exist_ok=True)

# Save model and scaler
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"Model saved to: {model_path}")
print(f"Scaler saved to: {scaler_path}")

---

## ✅ Demo Complete!

You can now:
1. Replace the sample data with your actual dataset
2. Adjust features and preprocessing as needed
3. Try different models
4. Use the saved model for inference